# Train Your Own Tokenizer: Byte-Pair Encoding

*Companion notebook for Chapter 31 of "Large Language Models from the Ground Up."*

This notebook builds a complete Byte-Pair Encoding (BPE) tokenizer from scratch — a trainer, an encoder, and a decoder — in pure Python, with no libraries to install. We train it on a paragraph of text, watch the merges it learns, and confirm it round-trips text perfectly.

BPE is the algorithm behind the tokenizers in GPT, Llama, Mistral, and essentially every modern LLM (as of 2026). The whole idea is four steps:

1. **Start** with every byte as its own token.
2. **Count** every adjacent pair of tokens.
3. **Merge** the most frequent pair into a new token.
4. **Repeat** until you've made enough merges.

Let's build it.

In [1]:
def get_pair_counts(ids):
    """Count every adjacent pair of token ids."""
    counts = {}
    for a, b in zip(ids, ids[1:]):
        counts[(a, b)] = counts.get((a, b), 0) + 1
    return counts

def apply_merge(ids, pair, new_id):
    """Replace every occurrence of `pair` with new_id."""
    out, i = [], 0
    while i < len(ids):
        if (i < len(ids) - 1
                and (ids[i], ids[i + 1]) == pair):
            out.append(new_id)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return out

print("helpers defined")

helpers defined


### The two helpers

`get_pair_counts` walks the token list in overlapping pairs and tallies each one — that's **step 2** of the algorithm. `apply_merge` walks the list and replaces every occurrence of a chosen pair with a single new token id, skipping two positions each time it merges — that's **step 3**.

### The trainer

`train_bpe` stitches the four steps together. It starts from raw **UTF-8 bytes** (step 1) — so every possible text is encodable and there is never an "unknown token." Each pass it recounts, picks the most frequent pair, and merges it, giving the new token an id of `256 + k` so learned tokens sit above the 256 raw byte values. The returned `merges` dictionary *is* the trained tokenizer; `vocab` maps every id back to the exact bytes it stands for.

In [2]:
def train_bpe(text, num_merges, verbose=False):
    """Learn `num_merges` BPE merges from `text`."""
    ids = list(text.encode("utf-8"))     # step 1: bytes
    vocab = {i: bytes([i]) for i in range(256)}
    merges = {}
    for k in range(num_merges):
        counts = get_pair_counts(ids)    # step 2
        if not counts:
            break
        pair = max(counts, key=counts.get)   # step 3
        new_id = 256 + k
        ids = apply_merge(ids, pair, new_id)
        merges[pair] = new_id
        vocab[new_id] = vocab[pair[0]] + vocab[pair[1]]
        if verbose:
            piece = vocab[new_id].decode("utf-8",
                                         "replace")
            print(f"merge {k+1:2d}: {piece!r:>12}"
                  f"  (count {counts[pair]}),"
                  f"  tokens now {len(ids)}")
    return merges, vocab

print("train_bpe defined")

train_bpe defined


### Train it on a paragraph

Watch the merges. The first ones are common letter pairs and short words ("th", "the", "the "); later merges build longer chunks out of the tokens made by earlier merges. Notice that spaces get absorbed into tokens — " the " becomes a single unit.

In [3]:
text = (
    "the quick brown fox jumps over the lazy dog. "
    "the dog was not amused, but the fox was quick "
    "to run over the hill and out of the story."
)
start = list(text.encode("utf-8"))
merges, vocab = train_bpe(text, 20, verbose=True)
print()
print("bytes at start :", len(start))

merge  1:         'th'  (count 6),  tokens now 127
merge  2:        'the'  (count 6),  tokens now 121
merge  3:       'the '  (count 6),  tokens now 115
merge  4:      ' the '  (count 5),  tokens now 110
merge  5:         ' o'  (count 4),  tokens now 106
merge  6:         'qu'  (count 2),  tokens now 104
merge  7:        'qui'  (count 2),  tokens now 102
merge  8:       'quic'  (count 2),  tokens now 100
merge  9:      'quick'  (count 2),  tokens now 98
merge 10:     'quick '  (count 2),  tokens now 96
merge 11:         'fo'  (count 2),  tokens now 94
merge 12:        'fox'  (count 2),  tokens now 92
merge 13:       'fox '  (count 2),  tokens now 90
merge 14:        ' ov'  (count 2),  tokens now 88
merge 15:       ' ove'  (count 2),  tokens now 86
merge 16:      ' over'  (count 2),  tokens now 84
merge 17: ' over the '  (count 2),  tokens now 82
merge 18:         'do'  (count 2),  tokens now 80
merge 19:        'dog'  (count 2),  tokens now 78
merge 20:         'wa'  (count 2),  tokens

### Encode and decode

Encoding new text starts from bytes and replays the learned merges **in the order they were learned** — this order matters, because later merges are built out of the tokens earlier merges created. Decoding glues each token's bytes back together and turns them into text.

In [4]:
def encode(text, merges):
    ids = list(text.encode("utf-8"))
    for pair, new_id in merges.items():
        ids = apply_merge(ids, pair, new_id)
    return ids

def decode(ids, vocab):
    data = b"".join(vocab[i] for i in ids)
    return data.decode("utf-8", errors="replace")

enc = encode(text, merges)
print("start bytes    :", len(start))
print("encoded tokens :", len(enc))
print("compression    :",
      round(len(start) / len(enc), 2), "x")
print("round trip ok  :", decode(enc, vocab) == text)

start bytes    : 133
encoded tokens : 76
compression    : 1.75 x
round trip ok  : True


### Round-trip on unseen text (with unseen characters)

The tokenizer was trained only on plain English, but because it works on **bytes**, it can encode and perfectly reconstruct text it never saw — including an accented "café" and characters outside its training set. There is no unknown token: the worst case is that a novel character falls back to its individual bytes.

In [5]:
sample = "the foxes jumped quickly! 42 caf\u00e9"
e = encode(sample, merges)
print("text   :", repr(sample))
print("bytes  :", len(sample.encode("utf-8")))
print("tokens :", len(e))
print("ids    :", e)
print("back   :", repr(decode(e, vocab)))
print("match  :", decode(e, vocab) == sample)

text   : 'the foxes jumped quickly! 42 café'
bytes  : 34
tokens : 25
ids    : [258, 267, 101, 115, 32, 106, 117, 109, 112, 101, 100, 32, 264, 108, 121, 33, 32, 52, 50, 32, 99, 97, 102, 195, 169]
back   : 'the foxes jumped quickly! 42 café'
match  : True


### What you built

That is a complete, correct BPE tokenizer — the same algorithm running inside the tokenizers of GPT, Llama, and Mistral, just with tens of thousands of merges instead of twenty. The trained `merges` dictionary is the whole thing; save it and you can encode any text, in any language, forever.

**Try it yourself:**
- Encode the single word `"strawberry"` and print each token's text piece with `decode([tok], vocab)`. You'll see the letters are fused inside a couple of chunks — that *is* the "how many r's" problem from Chapter 12.
- Retrain on a much larger paragraph and watch which whole words earn their own single token first.